# DSA 8301 — Statistical Inference for Big Data
**Kenya Housing Survey 2023/24 — Housing Financial Vulnerability**
Valerie Jerono | Reg. 222331 | Strathmore University iLabAfrica

This notebook starts from the full `master_frame.parquet` (21,347 rows x 443 columns)
with no manual pre-selection. Columns are observed, understood, and cleaned first;
the modelling set is then narrowed by measured importance against the target,
not by an upfront curated list. Markdown is kept minimal — explanation lives in
code comments and printed output.


In [ ]:
# 0.1  Environment setup (Colab-safe; skips gracefully on local Jupyter)
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except Exception as exc:
    IN_COLAB = False
    print(f'Not running in Colab (or mount skipped): {exc}')

!pip install -q pyarrow scipy statsmodels scikit-learn


In [ ]:
# 0.2  Imports and global display settings
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 120)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titleweight'] = 'bold'

RANDOM_STATE = 8301
rng = np.random.default_rng(RANDOM_STATE)

print('Environment ready.')


In [ ]:
# 0.3  Project paths. Same convention as the dissertation notebooks:
# Colab Drive if present, else current working directory.
DRIVE_ROOT = Path('/content/drive/MyDrive/KHS_Dissertation')
BASE = DRIVE_ROOT if DRIVE_ROOT.exists() else Path.cwd()
PQ   = BASE / 'data' / 'parquet'
FIGS = BASE / 'outputs' / 'figures' / 'dsa8301_v2'
TABS = BASE / 'outputs' / 'tables' / 'dsa8301_v2'

for folder in [PQ, FIGS, TABS]:
    folder.mkdir(parents=True, exist_ok=True)

print(f'BASE = {BASE}')
print(f'PQ   = {PQ}')
print(f'FIGS = {FIGS}')
print(f'TABS = {TABS}')


In [ ]:
# 1.1  Load the full master frame exactly as built (443 columns, no pre-selection)
candidates = [
    PQ / 'master_frame.parquet',
    Path('master_frame.parquet'),
    Path('/content/master_frame.parquet'),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        'Could not find master_frame.parquet. Place it in data/parquet, '
        'the working directory, or /content/.'
    )

df = pd.read_parquet(DATA_PATH)
print(f'Loaded: {DATA_PATH}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')


In [ ]:
# 1.2  First look: structure, not content. No column has been judged yet.
print('--- df.info() (truncated to dtype summary) ---')
df.info(memory_usage='deep', verbose=False)

print()
print('--- Dtype breakdown ---')
print(df.dtypes.value_counts().to_string())

print()
print(f'Memory footprint: {df.memory_usage(deep=True).sum() / 1e6:,.1f} MB')


In [ ]:
# 1.3  Identity and duplication checks. These must pass before anything else
# is trusted: if rows are not unique households, every later statistic is wrong.
id_col = 'interview__key' if 'interview__key' in df.columns else df.columns[0]

n_dupe_rows  = df.duplicated().sum()
n_dupe_ids   = df[id_col].duplicated().sum() if id_col in df.columns else np.nan

print(f'Candidate ID column: {id_col}')
print(f'Fully duplicated rows:        {n_dupe_rows}')
print(f'Duplicate {id_col} values:    {n_dupe_ids}')
print(f'Unique households:            {df[id_col].nunique():,}' if id_col in df.columns else '')

assert n_dupe_rows == 0, 'Full-row duplicates found — must resolve before proceeding.'


In [ ]:
# 1.4  First three rows, all columns visible in chunks of 40 so nothing is hidden
# behind pandas' default column truncation.
chunk = 40
for start in range(0, df.shape[1], chunk):
    cols = df.columns[start:start + chunk]
    print(f'--- columns {start} to {start + len(cols) - 1} ---')
    display(df[cols].head(3))


In [ ]:
# 2.1  Build a full column inventory: missingness, cardinality, dtype,
# and a few example values for every one of the 443 columns. Nothing is
# excluded at this stage regardless of how sparse or messy it looks.
inventory_rows = []
for c in df.columns:
    s = df[c]
    n_missing = s.isna().sum()
    pct_missing = n_missing / len(s) * 100
    n_unique = s.nunique(dropna=True)
    sample_vals = s.dropna().unique()[:4].tolist()
    inventory_rows.append({
        'column': c,
        'dtype': str(s.dtype),
        'n_missing': n_missing,
        'pct_missing': pct_missing,
        'n_unique_nonnull': n_unique,
        'sample_values': sample_vals,
    })

column_inventory = pd.DataFrame(inventory_rows).set_index('column')
column_inventory.to_csv(TABS / 'v2_full_column_inventory.csv')

print(f'Inventory built for all {len(column_inventory)} columns.')
print(f'Columns with 0% missing:   {(column_inventory.pct_missing == 0).sum()}')
print(f'Columns with >0% missing:  {(column_inventory.pct_missing > 0).sum()}')
print(f'Columns with >60% missing: {(column_inventory.pct_missing > 60).sum()}')
display(column_inventory.sort_values('pct_missing', ascending=False).head(15))


In [ ]:
# 2.2  Object-dtype columns deserve a closer look before anything downstream
# treats them as either clean strings or clean numbers. The dissertation
# work already found at least one landmine here: lp_* aggregate columns
# stored as object dtype with a literal 'No land' string mixed into
# otherwise-numeric values. We check every object column for this pattern,
# not just the ones already known about.
object_cols = df.select_dtypes(include='object').columns.tolist()
print(f'{len(object_cols)} object-dtype columns found.')

mixed_type_report = []
for c in object_cols:
    vals = df[c].dropna().unique()
    n_numeric_like = sum(str(v).replace('.', '', 1).replace('-', '', 1).isdigit() for v in vals)
    n_non_numeric  = len(vals) - n_numeric_like
    if n_numeric_like > 0 and n_non_numeric > 0:
        mixed_type_report.append({
            'column': c,
            'n_distinct_values': len(vals),
            'n_numeric_like': n_numeric_like,
            'n_non_numeric': n_non_numeric,
            'non_numeric_examples': [v for v in vals if not str(v).replace('.', '', 1).replace('-', '', 1).isdigit()][:5],
        })

mixed_type_df = pd.DataFrame(mixed_type_report)
print(f'Object columns mixing numeric-like and non-numeric values: {len(mixed_type_df)}')
if len(mixed_type_df) > 0:
    display(mixed_type_df)


In [ ]:
# 2.3  Sentinel-code detection. KHS uses 8/9/98/99 (and similar) as
# don't-know / refused / not-applicable codes on otherwise-numeric scales.
# These must be surfaced explicitly now so they are never silently
# averaged into a continuous variable later. We scan every numeric
# column for suspiciously common high integer codes relative to its
# own observed range.
SENTINEL_CANDIDATES = {8, 9, 98, 99, 999, -9, -8}

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
sentinel_hits = []
for c in numeric_cols:
    s = df[c].dropna()
    if len(s) == 0:
        continue
    present = sorted(set(s.unique()).intersection(SENTINEL_CANDIDATES))
    for code_val in present:
        n_hits = (s == code_val).sum()
        pct_of_nonnull = n_hits / len(s) * 100
        # only flag if the sentinel is rare relative to legitimate values,
        # i.e. it looks like a code, not a real measurement
        if 0 < pct_of_nonnull < 80:
            sentinel_hits.append({
                'column': c,
                'sentinel_code': code_val,
                'n_rows': n_hits,
                'pct_of_nonnull': pct_of_nonnull,
            })

sentinel_df = pd.DataFrame(sentinel_hits).sort_values('n_rows', ascending=False)
sentinel_df.to_csv(TABS / 'v2_sentinel_code_audit.csv', index=False)
print(f'{len(sentinel_df)} (column, sentinel-code) pairs flagged for review.')
display(sentinel_df.head(25))


In [ ]:
# 2.4  Sentinel codes are flagged, not yet removed. We hold them in a
# lookup so any later numeric computation can explicitly exclude them
# rather than relying on memory. This dict is consulted, not applied,
# until a specific analysis needs the column.
SENTINEL_MAP = {}
for _, row in sentinel_df.iterrows():
    SENTINEL_MAP.setdefault(row['column'], set()).add(row['sentinel_code'])

print(f'Sentinel lookup built for {len(SENTINEL_MAP)} columns.')
for c, codes_set in list(SENTINEL_MAP.items())[:10]:
    print(f'  {c}: {sorted(codes_set)}')


In [ ]:
# 3.1  KHS is a routed survey: rental-module questions (k-prefix) are only
# asked of renters, mortgage/ownership questions (l-prefix) only of owners,
# land-parcel questions (i-prefix / lp_*) only of landowners. A k-column
# at 99% missing is not a data quality failure — it is ~68% of households
# (owners) correctly skipping a renter-only question. We must check this
# before any missingness-based drop decision, or we will discard
# legitimate, informative variables.

# Routing flags, built from variables already in the registry.
routing_flags = pd.DataFrame(index=df.index)
if 'g02' in df.columns:
    routing_flags['is_renter'] = (df['g02'] == 1).astype('Int64')   # 1 = pays rent
if 'i00' in df.columns:
    routing_flags['owns_land'] = (df['i00'] == 1).astype('Int64')
if 'g03' in df.columns:
    routing_flags['is_owner'] = (df['g03'] == 1).astype('Int64')    # tenure type 1 = owner

print('Routing-flag prevalence (share of non-missing households):')
display(routing_flags.mean(numeric_only=True).rename('share').to_frame())


In [ ]:
# 3.2  For every high-missingness column, test whether its missingness
# lines up with a routing flag (module-conditional) rather than being
# scattered randomly across the whole sample. We do this by module
# prefix (k=rental, l=owner/mortgage, i/lp=land) since that is how KHS
# itself partitions the questionnaire.
def module_of(col):
    if col.startswith('k'):
        return 'k_rental'
    if col.startswith('l') and not col.startswith('lp_'):
        return 'l_owner_mortgage'
    if col.startswith('i') or col.startswith('lp_'):
        return 'i_land_parcel'
    return 'other'

high_missing_cols = column_inventory[column_inventory['pct_missing'] > 60].index.tolist()

module_missingness_check = []
for c in high_missing_cols:
    mod = module_of(c)
    pct_miss = column_inventory.loc[c, 'pct_missing']
    expected_routing_pct = np.nan
    if mod == 'k_rental' and 'is_renter' in routing_flags.columns:
        expected_routing_pct = (1 - routing_flags['is_renter'].mean()) * 100
    elif mod == 'l_owner_mortgage' and 'is_owner' in routing_flags.columns:
        expected_routing_pct = (1 - routing_flags['is_owner'].mean()) * 100
    elif mod == 'i_land_parcel' and 'owns_land' in routing_flags.columns:
        expected_routing_pct = (1 - routing_flags['owns_land'].mean()) * 100

    module_missingness_check.append({
        'column': c,
        'module': mod,
        'pct_missing': pct_miss,
        'expected_pct_if_routing': expected_routing_pct,
        'gap': abs(pct_miss - expected_routing_pct) if pd.notna(expected_routing_pct) else np.nan,
        'likely_structural': (pd.notna(expected_routing_pct) and abs(pct_miss - expected_routing_pct) < 15),
    })

module_missing_df = pd.DataFrame(module_missingness_check)
module_missing_df.to_csv(TABS / 'v2_module_conditional_missingness.csv', index=False)

n_structural = module_missing_df['likely_structural'].sum()
print(f'Of {len(module_missing_df)} high-missingness columns:')
print(f'  {n_structural} are consistent with module-conditional (structural) missingness')
print(f'  {len(module_missing_df) - n_structural} do not match a routing flag and need individual review')
display(module_missing_df.sort_values('module').head(25))


In [ ]:
# 3.3  Columns flagged as structurally missing are NOT dropped and are NOT
# treated as a data-quality problem. They get a categorical recoding plan
# instead: 'Not applicable' becomes its own explicit level rather than
# NaN, which preserves the information that the household was correctly
# routed away from the question. This recoding is deferred to section 5
# (cleaning), this cell only records which columns qualify.
structural_missing_cols = module_missing_df.loc[
    module_missing_df['likely_structural'], 'column'
].tolist()

print(f'{len(structural_missing_cols)} columns flagged for "Not applicable" recoding '
      f'rather than being dropped or imputed as if missing at random.')
print(structural_missing_cols[:20], '...' if len(structural_missing_cols) > 20 else '')


In [ ]:
# 4.1  Variable registry: documented type and role for the 125 columns
# already understood from prior cleaning work. type: continuous | ordinal
# | binary | categorical | id
VARIABLE_REGISTRY = {
    'interview__key': {'label': 'Household unique interview key', 'type': 'id', 'file': 'household'},
    'a01': {'label': 'County code (1-47)', 'type': 'categorical', 'file': 'household'},
    'countycode': {'label': 'County code string-padded (01-47)', 'type': 'categorical', 'file': 'household'},
    'a07_1': {'label': 'Urban/Rural stratum (1=Urban, 2=Rural)', 'type': 'binary', 'file': 'household'},
    'serial': {'label': 'KNBS household serial number', 'type': 'id', 'file': 'household'},
    'hhweight': {'label': 'Household survey weight', 'type': 'continuous', 'file': 'household'},
    'c01_1': {'label': 'Main drinking water source', 'type': 'ordinal', 'file': 'household'},
    'c01_2': {'label': 'Water collection method', 'type': 'ordinal', 'file': 'household'},
    'c01_3': {'label': 'Water treated before drinking', 'type': 'binary', 'file': 'household'},
    'c01_4': {'label': 'Time to water source (minutes, one way)', 'type': 'continuous', 'file': 'household'},
    'c02_1': {'label': 'Secondary drinking water source', 'type': 'ordinal', 'file': 'household'},
    'c04': {'label': 'Main toilet facility type', 'type': 'ordinal', 'file': 'household'},
    'c05': {'label': 'Handwashing facility available', 'type': 'binary', 'file': 'household'},
    'c07': {'label': 'Handwashing materials present', 'type': 'ordinal', 'file': 'household'},
    'c14_1': {'label': 'Monthly water expenditure (KES)', 'type': 'continuous', 'file': 'household'},
    'c10': {'label': 'Main lighting source', 'type': 'ordinal', 'file': 'household'},
    'c10_2': {'label': 'Hours of electricity supply per day', 'type': 'continuous', 'file': 'household'},
    'c10_4': {'label': 'Electricity connection type', 'type': 'binary', 'file': 'household'},
    'c11': {'label': 'Main cooking fuel', 'type': 'ordinal', 'file': 'household'},
    'c12': {'label': 'Primary cooking stove type', 'type': 'ordinal', 'file': 'household'},
    'c14_2': {'label': 'Monthly electricity expenditure (KES)', 'type': 'continuous', 'file': 'household'},
    'c14_3': {'label': 'Monthly other energy expenditure (KES)', 'type': 'continuous', 'file': 'household'},
    'c13__1': {'label': 'Owns radio', 'type': 'binary', 'file': 'household'},
    'c13__2': {'label': 'Owns mobile phone', 'type': 'binary', 'file': 'household'},
    'c13__3': {'label': 'Owns television', 'type': 'binary', 'file': 'household'},
    'c13__4': {'label': 'Owns computer/laptop', 'type': 'binary', 'file': 'household'},
    'c13__5': {'label': 'Owns motorcycle', 'type': 'binary', 'file': 'household'},
    'c13__6': {'label': 'Owns motor vehicle', 'type': 'binary', 'file': 'household'},
    'c13__7': {'label': 'Owns refrigerator', 'type': 'binary', 'file': 'household'},
    'internet': {'label': 'Household has internet access', 'type': 'binary', 'file': 'household'},
    'g01a': {'label': 'Monthly food expenditure (KES)', 'type': 'continuous', 'file': 'household'},
    'g01b': {'label': 'Monthly clothing expenditure (KES)', 'type': 'continuous', 'file': 'household'},
    'g01c': {'label': 'Monthly education expenditure (KES)', 'type': 'continuous', 'file': 'household'},
    'g01d': {'label': 'Monthly health expenditure (KES)', 'type': 'continuous', 'file': 'household'},
    'g01e': {'label': 'Monthly transport expenditure (KES)', 'type': 'continuous', 'file': 'household'},
    'g01f': {'label': 'Monthly communication expenditure (KES)', 'type': 'continuous', 'file': 'household'},
    'g01g': {'label': 'Monthly recreation expenditure (KES)', 'type': 'continuous', 'file': 'household'},
    'g01h': {'label': 'Monthly housing cost expenditure (KES)', 'type': 'continuous', 'file': 'household'},
    'g01i': {'label': 'Monthly energy expenditure (KES)', 'type': 'continuous', 'file': 'household'},
    'g01j': {'label': 'Monthly other expenditure (KES)', 'type': 'continuous', 'file': 'household'},
    'g01k': {'label': 'Monthly remittances sent (KES)', 'type': 'continuous', 'file': 'household'},
    'g02': {'label': 'Household pays rent', 'type': 'binary', 'file': 'household'},
    'g02_1': {'label': 'Monthly rent paid (KES)', 'type': 'continuous', 'file': 'household'},
    'g03': {'label': 'Housing tenure type', 'type': 'categorical', 'file': 'household'},
    'g04': {'label': 'Owns other property', 'type': 'binary', 'file': 'household'},
    'g05__1': {'label': 'Problem: overcrowding', 'type': 'binary', 'file': 'household'},
    'g05__2': {'label': 'Problem: poor water supply', 'type': 'binary', 'file': 'household'},
    'g05__3': {'label': 'Problem: poor sanitation', 'type': 'binary', 'file': 'household'},
    'g05__4': {'label': 'Problem: poor drainage', 'type': 'binary', 'file': 'household'},
    'g05__5': {'label': 'Problem: poor road access', 'type': 'binary', 'file': 'household'},
    'g05__6': {'label': 'Problem: insecurity', 'type': 'binary', 'file': 'household'},
    'g05__7': {'label': 'Problem: high rent/housing cost', 'type': 'binary', 'file': 'household'},
    'g05__8': {'label': 'Problem: poor structural condition', 'type': 'binary', 'file': 'household'},
    'h01': {'label': 'Perceived structural quality', 'type': 'ordinal', 'file': 'household'},
    'h02': {'label': 'Perceived roof quality', 'type': 'ordinal', 'file': 'household'},
    'h03': {'label': 'Perceived wall quality', 'type': 'ordinal', 'file': 'household'},
    'h04': {'label': 'Perceived floor quality', 'type': 'ordinal', 'file': 'household'},
    'h05': {'label': 'Perceived ventilation adequacy', 'type': 'ordinal', 'file': 'household'},
    'h06': {'label': 'Perceived natural lighting', 'type': 'ordinal', 'file': 'household'},
    'h07': {'label': 'Perceived water supply adequacy', 'type': 'ordinal', 'file': 'household'},
    'h08': {'label': 'Perceived sanitation adequacy', 'type': 'ordinal', 'file': 'household'},
    'h09': {'label': 'Perceived waste disposal', 'type': 'ordinal', 'file': 'household'},
    'h10': {'label': 'Perceived neighbourhood security', 'type': 'ordinal', 'file': 'household'},
    'h11': {'label': 'Overall housing satisfaction', 'type': 'ordinal', 'file': 'household'},
    'j04_1': {'label': 'Current tenure arrangement', 'type': 'binary', 'file': 'household'},
    'j05': {'label': 'Has formal title/ownership document', 'type': 'binary', 'file': 'household'},
    'j09': {'label': 'Housing cost is a financial burden', 'type': 'binary', 'file': 'household'},
    'j10': {'label': 'Ever missed rent/mortgage payment', 'type': 'binary', 'file': 'household'},
    'j11': {'label': 'At risk of eviction in next 12 months', 'type': 'binary', 'file': 'household'},
    'j12_1': {'label': 'Years in current dwelling', 'type': 'ordinal', 'file': 'household'},
    'j13': {'label': 'Satisfied with current tenure', 'type': 'binary', 'file': 'household'},
    'k02': {'label': 'Written tenancy agreement exists', 'type': 'binary', 'file': 'household'},
    'k05': {'label': 'Monthly rent (KES)', 'type': 'continuous', 'file': 'household'},
    'k09': {'label': 'Lease type', 'type': 'categorical', 'file': 'household'},
    'k21': {'label': 'Rent arrears status', 'type': 'ordinal', 'file': 'household'},
    'min_rent': {'label': 'Minimum rent in PSU (KES)', 'type': 'continuous', 'file': 'household'},
    'l07': {'label': 'Year dwelling was built', 'type': 'continuous', 'file': 'household'},
    'l13': {'label': 'Monthly mortgage/housing loan repayment (KES)', 'type': 'continuous', 'file': 'household'},
    'l14': {'label': 'Estimated market value of dwelling (KES)', 'type': 'continuous', 'file': 'household'},
    'l15': {'label': 'Imputed monthly housing cost/rent equivalent (KES)', 'type': 'continuous', 'file': 'household'},
    'l19': {'label': 'Plot/land size (decimal)', 'type': 'continuous', 'file': 'household'},
    'l21': {'label': 'Major renovation done', 'type': 'binary', 'file': 'household'},
    'e01': {'label': 'Solid waste disposal method', 'type': 'ordinal', 'file': 'household'},
    'e05': {'label': 'Proximity to waste dump/quarry', 'type': 'binary', 'file': 'household'},
    'e06': {'label': 'Flood exposure', 'type': 'ordinal', 'file': 'household'},
    'e07': {'label': 'Mudslide/erosion exposure', 'type': 'ordinal', 'file': 'household'},
    'e08': {'label': 'Terrain/slope type', 'type': 'ordinal', 'file': 'household'},
    'i00': {'label': 'Household owns land', 'type': 'binary', 'file': 'household'},
    'prop_util': {'label': 'Utilities as proportion of income', 'type': 'continuous', 'file': 'household'},
    'med_prop': {'label': 'Median utility-to-income ratio, county level', 'type': 'continuous', 'file': 'household'},
    'utilities': {'label': 'Household pays for utilities', 'type': 'binary', 'file': 'household'},
    'ctymin_ut': {'label': 'County-level minimum utility cost (KES)', 'type': 'continuous', 'file': 'household'},
    'med_brms': {'label': 'Median bedrooms in county', 'type': 'continuous', 'file': 'household'},
    'sf': {'label': 'Slum/informal settlement flag', 'type': 'binary', 'file': 'household'},
    'pln': {'label': 'Planning status of settlement', 'type': 'categorical', 'file': 'household'},
    'd01': {'label': 'Dwelling tenure type', 'type': 'categorical', 'file': 'dwelling'},
    'd03': {'label': 'Dwelling type', 'type': 'categorical', 'file': 'dwelling'},
    'd05': {'label': 'Located in approved building', 'type': 'binary', 'file': 'dwelling'},
    'd06': {'label': 'Building has planning approval', 'type': 'binary', 'file': 'dwelling'},
    'd07': {'label': 'Dwelling in hazard-prone area', 'type': 'binary', 'file': 'dwelling'},
    'd08': {'label': 'Outer wall material', 'type': 'ordinal', 'file': 'dwelling'},
    'd09': {'label': 'Roof material', 'type': 'ordinal', 'file': 'dwelling'},
    'd10': {'label': 'Floor material', 'type': 'ordinal', 'file': 'dwelling'},
    'd11': {'label': 'Number of rooms in dwelling', 'type': 'continuous', 'file': 'dwelling'},
    'd11_1': {'label': 'Floor area of dwelling (sq m)', 'type': 'continuous', 'file': 'dwelling'},
    'd11_2': {'label': 'Number of bedrooms', 'type': 'continuous', 'file': 'dwelling'},
    'd12': {'label': 'Number of rooms used for sleeping', 'type': 'continuous', 'file': 'dwelling'},
    'b04': {'label': 'Sex of household head', 'type': 'binary', 'file': 'individual'},
    'b05_years': {'label': 'Age in completed years', 'type': 'continuous', 'file': 'individual'},
    'b07': {'label': 'Marital status', 'type': 'categorical', 'file': 'individual'},
    'b10': {'label': 'Currently attending school', 'type': 'binary', 'file': 'individual'},
    'b11': {'label': 'Literacy status', 'type': 'binary', 'file': 'individual'},
    'ken_edu_isced11': {'label': 'Highest education level (ISCED-11)', 'type': 'ordinal', 'file': 'individual'},
    'any_disability': {'label': 'Any functional disability', 'type': 'binary', 'file': 'individual'},
    'resid': {'label': 'Residence type', 'type': 'binary', 'file': 'individual'},
    'hhsize': {'label': 'Total persons in household', 'type': 'continuous', 'file': 'individual'},
    'age_dep': {'label': 'Age dependency status', 'type': 'ordinal', 'file': 'individual'},
    'wap': {'label': 'Working-age population flag', 'type': 'binary', 'file': 'individual'},
    'inw': {'label': 'Individual survey weight', 'type': 'continuous', 'file': 'individual'},
    'i01_3': {'label': 'Land ownership type', 'type': 'categorical', 'file': 'land_parcels'},
    'i05': {'label': 'Land has title deed', 'type': 'ordinal', 'file': 'land_parcels'},
    'i06': {'label': 'Land use type', 'type': 'categorical', 'file': 'land_parcels'},
    'i08': {'label': 'Land dispute in last 5 years', 'type': 'binary', 'file': 'land_parcels'},
    'i10': {'label': 'Land registered', 'type': 'binary', 'file': 'land_parcels'},
    'i12': {'label': 'Land used as loan collateral', 'type': 'binary', 'file': 'land_parcels'},
}
print(f'Variable registry covers {len(VARIABLE_REGISTRY)} of {df.shape[1]} columns.')


In [ ]:
# 4.2  Apply the registry to the inventory, and be explicit about the
# remainder. Columns outside the registry are not assumed irrelevant —
# many are disaggregated (__N suffix) versions, county-derived rollups,
# or engineered fields from the same cleaning pipeline. They stay in
# scope and get triaged on missingness + naming pattern, not discarded.
column_inventory['registry_type'] = column_inventory.index.map(
    lambda c: VARIABLE_REGISTRY.get(c, {}).get('type', np.nan)
)
column_inventory['registry_label'] = column_inventory.index.map(
    lambda c: VARIABLE_REGISTRY.get(c, {}).get('label', np.nan)
)
column_inventory['in_registry'] = column_inventory['registry_type'].notna()

n_registered = column_inventory['in_registry'].sum()
n_unregistered = (~column_inventory['in_registry']).sum()
print(f'Registered columns:   {n_registered}')
print(f'Unregistered columns: {n_unregistered}')
print()
print('Unregistered columns by missingness tier:')
unreg = column_inventory.loc[~column_inventory['in_registry']]
tier = pd.cut(unreg['pct_missing'], bins=[-0.1, 0, 60, 100], labels=['Complete', 'Moderate (0-60%)', 'High (>60%)'])
print(tier.value_counts().to_string())


In [ ]:
# 4.3  Naming-pattern triage for the unregistered columns. KHS column
# prefixes map to questionnaire modules; disaggregated multi-select
# items use a "__N" suffix (e.g. g05__1..g05__8 are already registered
# as individual problem flags, but other __N groups are not). We group
# unregistered columns by prefix so the scale of each module is visible
# before deciding what, if anything, to fold in.
import re

def base_prefix(col):
    return re.split(r'__|_\d+$', col)[0]

unreg_cols = column_inventory.loc[~column_inventory['in_registry']].index
prefix_counts = pd.Series([base_prefix(c) for c in unreg_cols]).value_counts()
print('Unregistered columns grouped by base prefix (top 30):')
display(prefix_counts.head(30).to_frame('n_columns'))


In [ ]:
# 5.1  Fix the lp_* mixed-type columns found in 2.2. These store a
# literal 'No land' string alongside numeric-looking strings, which
# would silently become NaN (or worse, a parsing error) under a naive
# pd.to_numeric call. We split each into a numeric value plus an
# explicit 'has_land' indicator so the "no land" information survives
# as data rather than being discarded as a parsing failure.
df_clean = df.copy()

lp_cols = [c for c in df_clean.columns if c.startswith('lp_') and df_clean[c].dtype == object]
print(f'Fixing {len(lp_cols)} lp_* object columns: {lp_cols}')

for c in lp_cols:
    has_land_col = f'{c}_has_land'
    df_clean[has_land_col] = (df_clean[c] != 'No land').astype('Int64')
    df_clean[has_land_col] = df_clean[has_land_col].where(df_clean[c].notna(), pd.NA)
    df_clean[c] = pd.to_numeric(df_clean[c].replace('No land', np.nan), errors='coerce')

print('Post-fix dtype check:')
print(df_clean[lp_cols].dtypes)


In [ ]:
# 5.2  Apply the sentinel map from 2.4: replace flagged don't-know /
# refused / not-applicable codes with NaN in a dedicated working frame,
# while keeping a parallel "was_sentinel" flag so the distinction
# between "missing" and "respondent declined to answer" is not lost.
sentinel_flags_added = 0
for c, codes_set in SENTINEL_MAP.items():
    if c not in df_clean.columns:
        continue
    flag_col = f'{c}_was_sentinel'
    df_clean[flag_col] = df_clean[c].isin(codes_set).astype('Int64')
    df_clean.loc[df_clean[c].isin(codes_set), c] = np.nan
    sentinel_flags_added += 1

print(f'Sentinel codes neutralised to NaN in {sentinel_flags_added} columns.')
print(f'Working frame shape after sentinel + lp_* fixes: {df_clean.shape}')


In [ ]:
# 5.3  Structural missingness recoding (from section 3): for columns
# flagged as module-conditional, missingness is replaced with an
# explicit "Not applicable" category rather than left as NaN. This
# keeps "correctly skipped" distinct from "should have answered but
# data is absent" in every downstream summary and model.
recoded_count = 0
for c in structural_missing_cols:
    if c not in df_clean.columns:
        continue
    if pd.api.types.is_numeric_dtype(df_clean[c]):
        # cast to a string-categorical representation so 'Not applicable'
        # can coexist with numeric codes without forcing a dtype clash
        df_clean[c] = df_clean[c].astype('object')
    df_clean[c] = df_clean[c].where(df_clean[c].notna(), 'Not applicable')
    recoded_count += 1

print(f'{recoded_count} columns recoded with an explicit "Not applicable" level.')


In [ ]:
# 5.4  Consistency checks. These are sanity checks on logical relationships
# the questionnaire implies, not new cleaning steps -- failures here are
# reported, not silently auto-corrected, since "fixing" a logical
# inconsistency without knowing its source risks introducing new error.
consistency_issues = []

if {'g02', 'k05'}.issubset(df_clean.columns):
    renters_no_rent = ((df_clean['g02'] == 1) & (df_clean['k05'].isna())).sum()
    consistency_issues.append(('Renters (g02=1) with missing rent amount (k05)', renters_no_rent))

if {'i00', 'i05'}.issubset(df_clean.columns):
    no_land_has_title = ((df_clean['i00'] == 0) & (df_clean['i05'].notna()) & (df_clean['i05'] != 'Not applicable')).sum()
    consistency_issues.append(('No land (i00=0) but land title status recorded (i05)', no_land_has_title))

if {'b05_years'}.issubset(df_clean.columns):
    impossible_ages = ((df_clean['b05_years'] < 0) | (df_clean['b05_years'] > 120)).sum()
    consistency_issues.append(('Impossible ages (b05_years <0 or >120)', impossible_ages))

if {'d11', 'd11_2'}.issubset(df_clean.columns):
    more_bedrooms_than_rooms = (df_clean['d11_2'] > df_clean['d11']).sum()
    consistency_issues.append(('More bedrooms (d11_2) than total rooms (d11)', more_bedrooms_than_rooms))

consistency_df = pd.DataFrame(consistency_issues, columns=['check', 'n_flagged_rows'])
consistency_df.to_csv(TABS / 'v2_consistency_check_report.csv', index=False)
display(consistency_df)


In [ ]:
# 6.1  Housing Financial Vulnerability Score (HFVS): five pillars, each
# the mean of its available risk components, final score the mean of
# the five pillar scores. Convention: 0 = less vulnerable, 1 = more
# vulnerable. This formula is unchanged from the dissertation pipeline;
# what changes here is that it is now computed against the cleaned
# full-column frame rather than a pre-reduced one.

def to_risk(series, high_is_risk_values, scale_max=1):
    """Map a coded categorical/ordinal column to a 0-1 risk score."""
    s = pd.to_numeric(series, errors='coerce')
    if isinstance(high_is_risk_values, (list, set, tuple)):
        return s.isin(high_is_risk_values).astype(float)
    return (s / scale_max).clip(0, 1)

pillar_components = {}

# Pillar 1: Financial Stress
fin_cols = []
if 'j09' in df_clean.columns:
    fin_cols.append(to_risk(df_clean['j09'], {1}))
if 'j10' in df_clean.columns:
    fin_cols.append(to_risk(df_clean['j10'], {1}))
if 'j11' in df_clean.columns:
    fin_cols.append(to_risk(df_clean['j11'], {1}))
pillar_components['financial_stress'] = fin_cols

# Pillar 2: Tenure Insecurity
ten_cols = []
if 'j05' in df_clean.columns:
    ten_cols.append(to_risk(df_clean['j05'], {0}))
if 'j13' in df_clean.columns:
    ten_cols.append(to_risk(df_clean['j13'], {0}))
pillar_components['tenure_insecurity'] = ten_cols

# Pillar 3: Physical Hazard
haz_cols = []
if 'e06' in df_clean.columns:
    haz_cols.append(to_risk(df_clean['e06'], {1}))
if 'e07' in df_clean.columns:
    haz_cols.append(to_risk(df_clean['e07'], {1}))
if 'd07' in df_clean.columns:
    haz_cols.append(to_risk(df_clean['d07'], {1}))
pillar_components['physical_hazard'] = haz_cols

# Pillar 4: Structural/Quality Deprivation
qual_cols = []
for c in ['h01', 'h02', 'h03', 'h04']:
    if c in df_clean.columns:
        qual_cols.append(to_risk(df_clean[c], {3}))
pillar_components['structural_quality'] = qual_cols

# Pillar 5: Utility Deprivation
util_cols = []
if 'c01_3' in df_clean.columns:
    util_cols.append(to_risk(df_clean['c01_3'], {0}))
if 'c05' in df_clean.columns:
    util_cols.append(to_risk(df_clean['c05'], {0}))
if 'h07' in df_clean.columns:
    util_cols.append(to_risk(df_clean['h07'], {3}))
if 'h08' in df_clean.columns:
    util_cols.append(to_risk(df_clean['h08'], {3}))
pillar_components['utility_deprivation'] = util_cols

print('Pillar component counts (each should be >=1 for a usable pillar):')
for pillar, cols in pillar_components.items():
    print(f'  {pillar}: {len(cols)} component(s)')


In [ ]:
# 6.2  Build the five pillar scores and the final HFVS.
pillar_scores = pd.DataFrame(index=df_clean.index)
for pillar, cols in pillar_components.items():
    if len(cols) == 0:
        pillar_scores[pillar] = np.nan
        continue
    pillar_scores[pillar] = pd.concat(cols, axis=1).mean(axis=1, skipna=True)

df_clean['hfvs_score'] = pillar_scores.mean(axis=1, skipna=True)
for pillar in pillar_components:
    df_clean[f'pillar_{pillar}'] = pillar_scores[pillar]

print('HFVS profile:')
print(df_clean['hfvs_score'].describe().to_string())
print()
print(f'Households with complete HFVS (no missing pillars): {pillar_scores.notna().all(axis=1).sum():,}')


In [ ]:
# 6.3  Quick visual check: HFVS should be roughly continuous and not a
# degenerate spike at one value, since that would indicate a pillar
# construction error rather than genuine vulnerability variation.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df_clean['hfvs_score'].dropna(), bins=40, color='#2A7F62', edgecolor='black', alpha=0.8)
axes[0].set_title('HFVS Distribution')
axes[0].set_xlabel('HFVS score (0=least, 1=most vulnerable)')
axes[0].set_ylabel('Households')

pillar_scores.boxplot(ax=axes[1], rot=30)
axes[1].set_title('Pillar Score Spread')
axes[1].set_ylabel('Pillar score')

fig.tight_layout()
fig.savefig(FIGS / 'v2_hfvs_construction_check.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 7.1  Define the eligible candidate pool: every column except the
# target itself, its pillar components (which would trivially correlate
# since they are literally part of the target), identifiers, and raw
# survey weights (which are sampling-design artifacts, not predictors).
EXCLUDE_FROM_CANDIDATES = (
    ['hfvs_score'] + [c for c in df_clean.columns if c.startswith('pillar_')]
    + ['interview__key', 'serial', 'hhweight', 'inw', 'countycode']
)
candidate_cols = [c for c in df_clean.columns if c not in EXCLUDE_FROM_CANDIDATES]
print(f'Candidate pool for importance ranking: {len(candidate_cols)} columns '
      f'(out of {df_clean.shape[1]} total).')


In [ ]:
# 7.2  Split candidates into numeric-like and categorical-like by dtype
# and cardinality. A numeric column with very few distinct values
# (<=10) is treated as categorical/ordinal for testing purposes, since
# a Pearson correlation on a 3-level code is not meaningful.
numeric_candidates, categorical_candidates = [], []
for c in candidate_cols:
    s = df_clean[c]
    if pd.api.types.is_numeric_dtype(s) and s.nunique(dropna=True) > 10:
        numeric_candidates.append(c)
    else:
        categorical_candidates.append(c)

print(f'Numeric (continuous-like) candidates:    {len(numeric_candidates)}')
print(f'Categorical/ordinal/binary candidates:   {len(categorical_candidates)}')


In [ ]:
# 7.3  Numeric importance: Spearman correlation against HFVS (chosen
# over Pearson by default since many KHS monetary fields are skewed and
# HFVS itself is bounded; Spearman is monotonic-relationship robust).
numeric_importance = []
target = df_clean['hfvs_score']

for c in numeric_candidates:
    paired = pd.concat([df_clean[c], target], axis=1).dropna()
    if len(paired) < 30:
        continue
    rho, p = stats.spearmanr(paired[c], paired[target.name])
    numeric_importance.append({
        'variable': c,
        'n_obs': len(paired),
        'spearman_rho': rho,
        'abs_rho': abs(rho),
        'p_value': p,
        'significant_005': p < 0.05,
    })

numeric_importance_df = pd.DataFrame(numeric_importance).sort_values('abs_rho', ascending=False)
numeric_importance_df.to_csv(TABS / 'v2_numeric_importance_full.csv', index=False)
print(f'{len(numeric_importance_df)} numeric candidates tested.')
display(numeric_importance_df.head(25))


In [ ]:
# 7.4  Categorical importance: Kruskal-Wallis H-test of HFVS across the
# levels of each categorical/ordinal/binary candidate. Eta-squared
# (effect size) is reported alongside the p-value, since with n~21,000
# almost everything will be "significant" and effect size is what
# actually distinguishes a meaningful driver from background noise.
categorical_importance = []

for c in categorical_candidates:
    paired = pd.concat([df_clean[c], target], axis=1).dropna()
    if len(paired) < 30:
        continue
    groups = [g[target.name].values for _, g in paired.groupby(c) if len(g) >= 5]
    if len(groups) < 2:
        continue
    h_stat, p = stats.kruskal(*groups)
    n = len(paired)
    k = len(groups)
    eta_sq = (h_stat - k + 1) / (n - k) if n > k else np.nan
    categorical_importance.append({
        'variable': c,
        'n_obs': n,
        'n_levels_tested': k,
        'h_statistic': h_stat,
        'eta_squared': eta_sq,
        'p_value': p,
        'significant_005': p < 0.05,
    })

categorical_importance_df = pd.DataFrame(categorical_importance).sort_values('eta_squared', ascending=False)
categorical_importance_df.to_csv(TABS / 'v2_categorical_importance_full.csv', index=False)
print(f'{len(categorical_importance_df)} categorical candidates tested.')
display(categorical_importance_df.head(25))


In [ ]:
# 7.5  Combine into one ranked importance table on a common scale.
# Spearman |rho| and eta-squared are not the same statistic, but both
# sit on a roughly comparable 0-1 "share of association" footing, which
# is enough to rank for narrowing purposes -- the actual modelling
# stage will use proper coefficients, not this ranking, for inference.
combined_rank = pd.concat([
    numeric_importance_df.rename(columns={'abs_rho': 'effect_size'})[['variable', 'effect_size', 'p_value', 'significant_005']].assign(var_type='numeric'),
    categorical_importance_df.rename(columns={'eta_squared': 'effect_size'})[['variable', 'effect_size', 'p_value', 'significant_005']].assign(var_type='categorical'),
], ignore_index=True).sort_values('effect_size', ascending=False)

combined_rank.to_csv(TABS / 'v2_combined_importance_ranking.csv', index=False)
print(f'Full ranked candidate pool: {len(combined_rank)} variables.')
display(combined_rank.head(40))


In [ ]:
# 7.6  Progressive narrowing, shown as a funnel so the cut is visible
# rather than assumed. We narrow on a combination of statistical
# significance and a minimum effect-size floor, not on missingness
# alone -- a sparse-but-strong predictor (e.g. a hazard flag) should
# survive even if it has high missingness, provided that missingness is
# structural (section 3) and the signal is real.
funnel = []
funnel.append(('Full candidate pool', len(combined_rank)))

step1 = combined_rank[combined_rank['significant_005']]
funnel.append(('After: statistically significant (p<0.05)', len(step1)))

step2 = step1[step1['effect_size'] >= 0.02]
funnel.append(('After: effect size >= 0.02 (meaningful, not just detectable)', len(step2)))

step3 = step2.drop_duplicates(subset='variable')
funnel.append(('After: de-duplicated (some columns appear in both tests if borderline cardinality)', len(step3)))

funnel_df = pd.DataFrame(funnel, columns=['stage', 'n_variables'])
print(funnel_df.to_string(index=False))

MODEL_CANDIDATES = step3.sort_values('effect_size', ascending=False)
MODEL_CANDIDATES.to_csv(TABS / 'v2_model_candidate_shortlist.csv', index=False)
print()
print(f'Final shortlist carried into modelling: {len(MODEL_CANDIDATES)} variables.')
display(MODEL_CANDIDATES.head(50))


In [ ]:
# 7.7  Final split of the shortlist into continuous and categorical
# lists, named to match the convention used in the modelling stage
# (MODEL_CONTINUOUS / MODEL_CATEGORICAL), so this notebook plugs
# directly into the next stage of the pipeline.
MODEL_CONTINUOUS = MODEL_CANDIDATES.loc[MODEL_CANDIDATES['var_type'] == 'numeric', 'variable'].tolist()
MODEL_CATEGORICAL = MODEL_CANDIDATES.loc[MODEL_CANDIDATES['var_type'] == 'categorical', 'variable'].tolist()

print(f'MODEL_CONTINUOUS:  {len(MODEL_CONTINUOUS)} variables')
print(f'MODEL_CATEGORICAL: {len(MODEL_CATEGORICAL)} variables')
print()
print('Continuous:', MODEL_CONTINUOUS)
print()
print('Categorical:', MODEL_CATEGORICAL)


In [ ]:
# 7.8  Save the cleaned, importance-ranked working frame so the next
# notebook stage (parametric + nonparametric testing) can load it
# directly without repeating the cleaning and ranking work above.
model_df_cols = ['hfvs_score'] + [c for c in df_clean.columns if c.startswith('pillar_')] + MODEL_CONTINUOUS + MODEL_CATEGORICAL
model_df_cols = list(dict.fromkeys(model_df_cols))  # de-dupe, preserve order
model_df = df_clean[model_df_cols].copy()

model_df.to_parquet(PQ / 'model_df_v2.parquet')
print(f'Saved model_df_v2.parquet: {model_df.shape[0]:,} rows x {model_df.shape[1]} columns')
print(f'Location: {PQ / "model_df_v2.parquet"}')


In [ ]:
# 8.1  Stage summary: what this notebook established and what carries
# forward into the next stage (parametric + nonparametric testing).
print('=== STAGE 1 SUMMARY ===')
print(f'Source frame:        {DATA_PATH.name} ({df.shape[0]:,} rows x {df.shape[1]} columns)')
print(f'Registry coverage:   {n_registered} of {df.shape[1]} columns documented')
print(f'Sentinel pairs found: {len(sentinel_df)} (column, code) combinations neutralised')
print(f'Structural-missing columns recoded: {len(structural_missing_cols)}')
print(f'lp_* mixed-type columns fixed:      {len(lp_cols)}')
print(f'HFVS households with complete pillars: {pillar_scores.notna().all(axis=1).sum():,}')
print(f'Candidates tested for importance:   {len(combined_rank)}')
print(f'Final modelling shortlist:          {len(MODEL_CANDIDATES)} '
      f'({len(MODEL_CONTINUOUS)} continuous, {len(MODEL_CATEGORICAL)} categorical)')
print()
print('Saved outputs:')
for f in sorted(TABS.glob('v2_*.csv')):
    print(f'  {f.name}')
print(f'  {PQ / "model_df_v2.parquet"}')
print()
print('Next stage: EDA, normality testing, Part C parametric methods, '
      'Part D nonparametric methods, using model_df_v2.parquet as the input.')
